# 10 самых длинных GPS-треков

Длина — сумма расстояний между последовательными точками `/sensing/gnss/master/fix` до прореживания для карты. GPS-скачки с вычисленной скоростью свыше 30 м/с исключаются перед подсчётом длины.


In [6]:
import sys
from pathlib import Path

NOTEBOOKS_DIR = Path.cwd() / "notebooks" if (Path.cwd() / "notebooks").is_dir() else Path.cwd()
sys.path.insert(0, str(NOTEBOOKS_DIR))

from tools.gps_tracks import NoGpsDataError, load_gps_track
from tools.track_map import show_tracks_map

DATA_DIR = next(
    (candidate for candidate in (Path.cwd() / "data", Path.cwd().parent / "data") if candidate.is_dir()),
    None,
)
if DATA_DIR is None:
    raise FileNotFoundError("Не найден каталог data/.")


In [7]:
candidates = []
for bag_dir in sorted(DATA_DIR.iterdir()):
    if not bag_dir.is_dir():
        continue
    try:
        track = load_gps_track(DATA_DIR, bag_dir.name, max_speed_m_s=30)
    except NoGpsDataError:
        continue
    if track["point_count"] >= 2:
        candidates.append(track)

top_tracks = sorted(candidates, key=lambda track: track["length_m"], reverse=True)[:10]
if len(top_tracks) < 10:
    raise ValueError(f"Найдено только {len(top_tracks)} GPS-треков с двумя и более точками")

print(" №  ID                    Длина, км  GPS-точек")
for rank, track in enumerate(top_tracks, start=1):
    print(f"{rank:2}  {track['id']:<20}  {track['length_m'] / 1000:9.2f}  {track['point_count']:10}")


 №  ID                    Длина, км  GPS-точек
 1  30618_27e994fc             8.10       18250
 2  30618_88548b02             6.22       12344
 3  30639_d3c43d69             6.02       10437
 4  30639_0be558e2             5.95       12728
 5  30618_68d1748a             5.86       12303
 6  30618_9c09b081             5.85       13169
 7  30618_2dbce472             5.83       13478
 8  30618_defd0170             5.81        9540
 9  30618_0652866c             5.76       14557
10  30618_ab5921a4             5.76       11425


In [8]:
from tools.path_graphs import load_path_graph_tracks

graph_tracks = load_path_graph_tracks(DATA_DIR.parent / "path_graphs")
graph_map = show_tracks_map(graph_tracks + top_tracks, height=600)


Map(center=[55.80482856166667, 37.425154245], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom…